# Week 6 - Tensor Cores and MMA Programming (Project Playbook)

This notebook is a practical execution guide for **Mini-Project 6**. It is intentionally written as a workflow, not just a concept summary: what to implement, how to measure, how to validate, what can break, and how to report results.

## Mini-Project Target
Implement Tensor Core GEMM using WMMA for FP16 and TF32, then evaluate:

1. Throughput (TFLOPS achieved).
2. Numerical error vs FP32 cuBLAS baseline.
3. Loss-scaling impact on tiny gradient survival.

## Why This Project Matters

Modern training stacks rely on mixed precision because Tensor Cores massively increase arithmetic density. That speedup is only useful if you understand where precision is lost and how to keep training stable.

The practical debugging questions this lab prepares you for are:

- Is my kernel actually using Tensor Cores?
- Why is speed low even though Tensor Cores are enabled?
- Why does a model stop learning under reduced precision?
- Is the error coming from multiply precision, accumulation precision, or storage precision?

## Concept Core: Warp-Level WMMA

A WMMA operation is a **warp contract**. One warp (32 threads) cooperates to process one matrix tile operation.

For a $16 \times 16$ tile:

- Tile elements: $256$
- Threads in a warp: $32$
- Elements handled per thread (conceptually): $256 / 32 = 8$

All WMMA functions end in `_sync` because all lanes must participate in lockstep. Divergence at this stage breaks the collective assumption.

## Precision Ladder and Hardware Density

- **FP16**: 16-bit format (5 exponent, 10 mantissa).
- **TF32**: FP32-like range with truncated mantissa for Tensor Core math path.
- **FP8**: much smaller format, higher density, lower precision margin.

GEMM is usually compute-bound because data is reused across many fused multiply-add operations. Lower precision means smaller multiplier circuits, so more arithmetic units can fit on-die, increasing theoretical peak throughput.

## Project Architecture in This Repo

- `src/main.cu`: host orchestration, data setup, cuBLAS baseline, timing, and error stats.
- `src/wmma_fp16.cu`: FP16 WMMA kernel (`16x16x16`).
- `src/wmma_tf32.cu`: TF32 WMMA kernel (`16x16x8`).
- `analysis/precision_loss.py`: underflow and loss-scaling analysis.

Baseline matrix shape used in harness: $M=4096, N=4096, K=11008$.

## Implementation Milestones (Do This In Order)

1. Confirm build works and program runs once end-to-end.
2. Validate FP16 WMMA path for correctness on small test dimensions first.
3. Enable TF32 path and verify architecture target is Ampere+ (`sm_80+`).
4. Benchmark on full shape and compute TFLOPS.
5. Compare numerical error against cuBLAS FP32 reference.
6. Run precision-loss analysis and explain underflow + scaling behavior.
7. Produce a short findings report with limitations and next optimizations.

## Build and Run Commands (Student Workflow)

Run from repository root:

```bash
cmake -S . -B build -DCMAKE_CUDA_ARCHITECTURES=80
cmake --build build -j
./build/week6_tensor_cores
```

When to rerun each command:

- Re-run configure if build settings change or `build/` is recreated.
- Re-run build after source edits.
- Re-run executable whenever you need fresh benchmark/error output.

## Converting ms to TFLOPS

Your binary prints runtime in milliseconds.
You need a formula to interpret that runtime.

$$\text{TFLOPS} = \frac{2MNK}{t \cdot 10^{12}}$$

Where:
- $M, N, K$ are matrix dimensions
- $t$ is execution time in seconds

In [ ]:
def tflops(m, n, k, milliseconds):
    seconds = milliseconds / 1000.0
    return (2.0 * m * n * k) / (seconds * 1.0e12)

M, N, K = 4096, 4096, 11008
example_ms = 3.0
print(f'Example throughput at {example_ms} ms: {tflops(M, N, K, example_ms):.2f} TFLOPS')

## WMMA Kernel Mental Model (What Each Block Computes)

In `wmma_fp16.cu`, each block has one warp (`dim3 block(32)`), and each block maps to one output tile:

- `row = blockIdx.y * 16`
- `col = blockIdx.x * 16`

These are **tile start coordinates**, not global totals. The kernel then loops over K in tile-sized chunks and accumulates into one output fragment before storing to C.

In [ ]:
tile_m, tile_n = 16, 16
for by in range(2):
    for bx in range(3):
        row = by * tile_m
        col = bx * tile_n
        print(f'block ({bx},{by}) -> output tile starts at row={row}, col={col}')

## Required Experiments for Mini-Project 6

### A) Throughput
Run executable and collect runtime for cuBLAS, WMMA TF32, and WMMA FP16.

### B) Numerical Error
Run repeated randomized inputs and summarize:

- max absolute difference
- relative L2
- mean and percentile statistics across trials

### C) Loss Scaling
Use the analysis script to show tiny-gradient collapse without scaling and partial recovery with scale factor (for example 1024).

In [ ]:
from pathlib import Path
import subprocess

script = Path('analysis/precision_loss.py')
result = subprocess.run(['python3', str(script)], capture_output=True, text=True, check=True)
print(result.stdout)

## 1000-Trial Accuracy Protocol (Suggested)

1. Generate 1000 random matrix pairs with fixed seed control.
2. For each trial, run reference cuBLAS FP32 and candidate WMMA path.
3. Record max-abs and relative-L2 per trial.
4. Report mean, std, p50, p90, p99, and worst-case.

This is more informative than a single run because distribution tails reveal numerical risk.

In [ ]:
import math

def summary_stats(values):
    values = sorted(values)
    n = len(values)
    def q(p):
        idx = min(n - 1, max(0, int(round((n - 1) * p))))
        return values[idx]
    mean = sum(values) / n
    var = sum((v - mean) ** 2 for v in values) / n
    return {
        'mean': mean,
        'std': math.sqrt(var),
        'p50': q(0.50),
        'p90': q(0.90),
        'p99': q(0.99),
        'max': values[-1],
    }

toy = [0.001, 0.002, 0.0008, 0.0032, 0.0011]
print(summary_stats(toy))

## Debugging Checklist (When Results Look Wrong)

- Build target architecture is Ampere+ (`sm_80`) for TF32 WMMA.
- Matrix layouts match kernel expectations (A row-major, B col-major for this implementation).
- Tile dimensions match WMMA shape constraints (FP16 `16x16x16`, TF32 `16x16x8`).
- Leading dimension passed to `load_matrix_sync` is correct.
- Grid dimensions cover full output matrix.
- Boundary guard is present for edge tiles.
- Timing uses CUDA events around kernel launch and stream sync.
- Reference and candidate outputs are copied back before error metrics.

## Deliverable Template

Use this section structure in your report or README:

1. **Implementation**: FP16 WMMA kernel + TF32 WMMA kernel details.
2. **Benchmark Setup**: GPU model, CUDA version, matrix dimensions, trial count.
3. **Performance Results**: runtime and TFLOPS table vs cuBLAS baseline.
4. **Accuracy Results**: max-abs / relative-L2 distributions over trials.
5. **Loss Scaling Findings**: underflow behavior and scaled recovery.
6. **Discussion**: bottlenecks, what is missing from production GEMM (shared memory staging, pipelining, multistage tiling), and next optimization steps.

## Stretch Goals (Optional)

- Add multi-warp threadblock tiles and shared-memory staging.
- Compare arithmetic intensity and memory traffic between naive and tiled designs.
- Add a baseline using cuBLAS Tensor Op math mode for direct comparison.
- Add a compact benchmark script to automate repeated runs and CSV output.